## 1. Imports

In [1]:
import pandas as pd
import random
import re
import os
from sklearn.model_selection import train_test_split
from datasets import load_dataset

random.seed(42)

c:\Users\Tatha\anaconda3\envs\lead-routing\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
load_dotenv()

## If you do not have open AI key use the below Huggingface embedding
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")

In [3]:
# 1. Load the dataset from HF library
raw_dataset = load_dataset("Tobi-Bueck/customer-support-tickets")

# 2. Extract the 'train' split into a Pandas DataFrame properly
df = pd.DataFrame(raw_dataset['train'])


In [4]:
display(df.head())

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


In [5]:
print(df['language'].value_counts())

language
de    33504
en    28261
Name: count, dtype: int64


In [6]:
df = df[df['language'] == 'en'].reset_index(drop=True)

In [7]:
col1 = 'subject'      
col2 = 'body'   
new_col = 'query'       

# 2. Merge the strings with a space in between
df[new_col] = df[col1].astype(str) + " " + df[col2].astype(str)

# 3. Drop the original two columns and overwrite df
df = df.drop(columns=[col1, col2])


In [8]:
display(df.head())

,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,query
0,"Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN,"Account Disruption Dear Customer Support Team,..."
1,Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN,Query About Smart Home System Integration Feat...
2,We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN,Inquiry Regarding Invoice Details Dear Custome...
3,Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN,Question About Marketing Agency Software Compa...
4,Thank you for your inquiry. Please specify whi...,Request,Technical Support,high,en,51.0,Feature,Product,Documentation,Feedback,NaN,NaN,NaN,NaN,"Feature Query Dear Customer Support,\n\nI hope..."


In [9]:
columns_to_drop = ['answer','priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3' ,'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']

In [10]:
df = df.drop(columns=columns_to_drop)

In [12]:
df=df.dropna(subset=['query']).reset_index(drop=True)

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24621 entries, 0 to 24620
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   type    24621 non-null  str  
 1   queue   24621 non-null  str  
 2   query   24621 non-null  str  
dtypes: str(3)
memory usage: 10.7 MB


In [14]:
df.head()

,type,queue,query
0,Incident,Technical Support,"Account Disruption Dear Customer Support Team,..."
1,Request,Returns and Exchanges,Query About Smart Home System Integration Feat...
2,Request,Billing and Payments,Inquiry Regarding Invoice Details Dear Custome...
3,Problem,Sales and Pre-Sales,Question About Marketing Agency Software Compa...
4,Request,Technical Support,"Feature Query Dear Customer Support,\n\nI hope..."


In [15]:
print(df['type'].value_counts())

type
Incident    9783
Request     7142
Problem     5143
Change      2553
Name: count, dtype: int64


In [16]:
print(df['queue'].value_counts())

queue
Technical Support                  7090
Product Support                    4643
Customer Service                   3755
IT Support                         2883
Billing and Payments               2543
Returns and Exchanges              1216
Service Outages and Maintenance     996
Sales and Pre-Sales                 692
Human Resources                     468
General Inquiry                     335
Name: count, dtype: int64


In [30]:
# dropping HR tickets — internal employee queries, not customer-facing
df = df[df['queue'] != 'Human Resources'].reset_index(drop=True)

In [31]:

# merging queue labels into 4 business teams
queue_merge_map = {
    'Technical Support':              'Tech Support',
    'Service Outages and Maintenance':'Tech Support',
    'IT Support':                     'Tech Support',

    'Customer Service':               'Customer Service',
    'Returns and Exchanges':          'Customer Service',
    'Product Support':                'Customer Service',

    'Billing and Payments':           'Billing',
    'General Inquiry':                'Billing',

    'Sales and Pre-Sales':            'Sales',
}

df['queue'] = df['queue'].map(queue_merge_map)

In [32]:
unmapped = df['queue'].isna().sum()

In [33]:
print(df['queue'].value_counts())

queue
Tech Support        10969
Customer Service     9614
Billing              2878
Sales                 692
Name: count, dtype: int64


In [34]:
df.to_csv('../data/cleaned/filtered_business_leads.csv', index=False)